In [1]:
import matplotlib.pyplot as plt
tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "times",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 18,
    "font.size": 18,
    # Make the legend/bel fonts a little smaller
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
}

plt.rcParams.update(tex_fonts)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Standalone CAC Saint-Venant DeepONet dataset generator.

Both initial water-depth and discharge profiles are generated by truncated
Chebyshev expansions around steady reference profiles. The steady reference discharge is fixed at Qbar(x)=11.93 m^3/s. The steady
water-depth profile Hbar(x) is obtained by integrating the steady Saint-Venant
ODE backward from the downstream equilibrium boundary:

    H0(x) = Hbar(x) + sum_j a_j T_j(2x/L - 1)
    Q0(x) = Qbar(x) + sum_j b_j T_j(2x/L - 1)

The continuity equation satisfies dQbar/dx=0, hence Qbar remains spatially
constant at 11.93 m^3/s. The collected DeepONet input is the physical
state itself, not the normalized error. The default Chebyshev amplitudes are A_H=0.3 m and A_Q=2.5 m^3/s.
A trajectory is written only after it reaches the prescribed horizon without
divergence; any divergent trajectory is discarded in full.


    x.dat = [H_1, ..., H_N, Q_1, ..., Q_N]

The environment may still internally use normalized-error observations for its
own interface; those observations are not written to x.dat.

Saved files:
    x.dat          : physical state concat([H, Q])
    y.dat          : normalized action label a in [-1, 1]
    y_ul.dat       : clipped physical gate U_L
    y_ul_raw.dat   : raw backstepping gate before clipping
    y_ubs.dat      : characteristic backstepping command U_bs(t)
    meta.dat       : episode_id, time_s
    grid.dat       : spatial grid X
    hbar_qbar.dat  : concat([Hbar, Qbar])
    rejected_episodes.dat : attempt_id, seed, failure_time, diagnostic values

Run example:
    python collect_deeponet_data_chebyshev_fixed_Qbar_11p93_ODE_Hbar_AH03_AQ25_discard_diverged.py --num 1000 --seed 40
"""

from __future__ import annotations

import argparse
import datetime
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
from scipy.integrate import solve_ivp

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

from cac_saint_venant_env_no_bs import (
    CACParams,
    CACSaintVenantPDE1D,
    source,
)


# ============================================================
# 1. SAC-matched settings
# ============================================================

N_DEFAULT = 102
T_DEFAULT = 40000.0
DT_DEFAULT = 5.0
CONTROL_SAMPLE_RATE_DEFAULT = 400.0

MAX_STATE_VALUE = 1e6

# Current SAC gate range requested by the user.
MIN_CONTROL_VALUE = 0.15
MAX_CONTROL_VALUE = 0.4

H_SCALE = 0.10
Q_SCALE = 3.00
OBS_CLIP = 50.0

CFL = 0.45
CHECK_CFL = True
WELL_BALANCED = True

# Truncated Chebyshev initial-condition sampler.
# The amplitudes are maximum interior deviations after normalization.
CHEB_ORDER_DEFAULT = 6
H_CHEB_AMPLITUDE_DEFAULT = 0.30
Q_CHEB_AMPLITUDE_DEFAULT = 2.50
CHEB_COEFF_DECAY_DEFAULT = 2.0
MAX_ATTEMPT_FACTOR_DEFAULT = 20
Q_BAR_FIXED_DEFAULT = 11.93


# ============================================================
# 2. CAC parameters, grid, reference, initial conditions
# ============================================================

def make_cac_table_params(
    q_bar: float = Q_BAR_FIXED_DEFAULT,
    h_bar_0: float = 3.80,
    h_bar_L: float = 4.70,
) -> CACParams:
    """Create CAC parameters.

    ``q_bar`` is the prescribed spatially constant reference discharge.
    The steady water-depth profile is subsequently obtained from the ODE.
    """
    return CACParams(
        L=11239.0,
        B=40.0,
        g=9.81,
        I=7.92e-5,
        K=33.0,
        mu0=0.4,
        muL=0.4,
        z_up=5.00,
        U0_bar=0.1548,
        U_L_bar=0.30,
        h_s=4.094,
        Q_bar=float(q_bar),
        H_bar_0=float(h_bar_0),
        H_bar_L=float(h_bar_L),
        H_min=0.20,
        max_state_value=1e10,
    )


def make_grid(p: CACParams, N: int) -> Tuple[np.ndarray, float]:
    X = np.linspace(0.0, p.L, N)
    dx = p.L / (N - 1)
    return X, dx


def downstream_equilibrium_depth_from_q(Q: float, p: CACParams) -> float:
    """Return Hbar(L) from the downstream equilibrium gate relation."""
    Q = float(Q)
    if Q <= 0.0:
        raise ValueError("The equilibrium discharge must be positive.")
    kinetic_head = (
        Q**2 / (2.0 * p.g * p.B**2 * p.muL**2)
    ) ** (1.0 / 3.0)
    return float(p.h_s + p.U_L_bar + kinetic_head)


def upstream_equilibrium_discharge_from_h(H0: float, p: CACParams) -> float:
    """Return the upstream equilibrium discharge for a given Hbar(0)."""
    head = float(p.z_up) - float(H0)
    if head <= 0.0:
        raise ValueError("Upstream equilibrium requires z_up - Hbar(0) > 0.")
    return float(p.mu0 * p.B * p.U0_bar * np.sqrt(2.0 * p.g * head))


def integrate_steady_fixed_q_ode(
    X: np.ndarray,
    p: CACParams,
    Q_candidate: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Integrate the steady Saint-Venant ODE backward from x=L.

    With no lateral inflow, the steady equations are

        dHbar/dx = g B Hbar (I - Jbar)
                    / (g B Hbar - Qbar^2/(B Hbar^2)),
        dQbar/dx = 0.

    Although Qbar is constant along x, it is retained as the second ODE state so the steady pair is returned in a uniform form.
    Its constant value is prescribed by ``Q_candidate``.
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 1 or len(X) < 2:
        raise ValueError("X must be a one-dimensional grid with at least two points.")
    if not np.all(np.diff(X) > 0.0):
        raise ValueError("X must be strictly increasing.")

    Q_candidate = float(Q_candidate)
    Hbar_L = downstream_equilibrium_depth_from_q(Q_candidate, p)

    def rhs(_x: float, state: np.ndarray) -> np.ndarray:
        H = float(state[0])
        Q = float(state[1])
        if not np.isfinite(H) or not np.isfinite(Q):
            raise FloatingPointError("Non-finite state in steady reference ODE.")
        if H <= float(p.H_min):
            raise FloatingPointError("Steady reference ODE violates minimum depth.")

        area = p.B * H
        hydraulic_radius = p.B * H / (p.B + 2.0 * H)
        friction_slope = Q * abs(Q) / (
            p.K**2 * area**2 * hydraulic_radius ** (4.0 / 3.0)
        )
        denominator = p.g * p.B * H - Q**2 / (p.B * H**2)
        if abs(denominator) <= 1e-10:
            raise FloatingPointError(
                "Steady ODE approaches the critical-flow singularity."
            )

        dHdx = p.g * p.B * H * (p.I - friction_slope) / denominator
        dQdx = 0.0
        if not np.isfinite(dHdx):
            raise FloatingPointError("Non-finite dHbar/dx in steady reference ODE.")
        return np.array([dHdx, dQdx], dtype=float)

    X_descending = X[::-1]
    sol = solve_ivp(
        fun=rhs,
        t_span=(float(X[-1]), float(X[0])),
        y0=np.array([Hbar_L, Q_candidate], dtype=float),
        t_eval=X_descending,
        rtol=1e-9,
        atol=1e-11,
        max_step=p.L / 1000.0,
    )

    if not sol.success:
        raise RuntimeError(f"Failed to solve steady ODE: {sol.message}")
    if sol.y.shape[1] != len(X):
        raise RuntimeError("Fixed-Q steady ODE did not return all grid points.")

    Hbar = np.asarray(sol.y[0][::-1], dtype=float)
    Qbar = np.asarray(sol.y[1][::-1], dtype=float)
    if not np.all(np.isfinite(Hbar)) or not np.all(np.isfinite(Qbar)):
        raise FloatingPointError("Steady reference contains NaN or Inf.")
    if float(np.min(Hbar)) <= float(p.H_min):
        raise RuntimeError("Steady Hbar profile violates the minimum depth.")

    return Hbar, Qbar


def solve_steady_reference_ode(
    X: np.ndarray,
    p: CACParams,
    q_bar_fixed: float = Q_BAR_FIXED_DEFAULT,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Solve Hbar(x) for a prescribed constant reference discharge.

    With no lateral inflow, the steady continuity equation gives
    dQbar/dx=0. Therefore Qbar(x) is fixed to ``q_bar_fixed`` at every
    spatial location. The downstream gate equilibrium relation determines
    Hbar(L), after which the steady momentum ODE is integrated backward to
    obtain Hbar(x). No shooting method is used.
    """
    q_bar_fixed = float(q_bar_fixed)
    if not np.isfinite(q_bar_fixed) or q_bar_fixed <= 0.0:
        raise ValueError("q_bar_fixed must be a positive finite number.")

    Hbar, Qbar = integrate_steady_fixed_q_ode(
        X=X,
        p=p,
        Q_candidate=q_bar_fixed,
    )

    # Numerical integration retains Qbar as the second ODE state, but enforce
    # the prescribed constant exactly to avoid any round-off drift.
    Qbar = np.full_like(X, q_bar_fixed, dtype=float)
    return Hbar, Qbar, q_bar_fixed


def make_reference_profiles(
    X: np.ndarray,
    p: CACParams,
    reference: str = "ode",
    q_bar_fixed: float = Q_BAR_FIXED_DEFAULT,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """Construct reference profiles with fixed spatially constant Qbar."""
    if reference in {"ode", "strict"}:
        return solve_steady_reference_ode(X, p, q_bar_fixed=q_bar_fixed)
    if reference == "linear":
        Hbar = p.H_bar_0 + (p.H_bar_L - p.H_bar_0) * (X / p.L)
        Qbar = np.full_like(X, q_bar_fixed, dtype=float)
        return Hbar, Qbar, float(q_bar_fixed)
    raise ValueError("reference must be 'ode', 'strict', or 'linear'.")


def make_fixed_initial_condition(X: np.ndarray, p: CACParams) -> Tuple[np.ndarray, np.ndarray]:
    # Match SAC fixed test initial condition.
    H0 = 3.75 + (4.65 - 3.75) * (X / p.L)
    Q0 = np.full_like(X, 10.0)
    return H0, Q0


def sample_truncated_chebyshev(
    xi: np.ndarray,
    rng,
    order: int,
    max_amplitude: float,
    coefficient_decay: float,
) -> np.ndarray:
    """Sample a smooth truncated Chebyshev perturbation.

    A random coefficient is assigned to every retained Chebyshev mode. Higher
    modes are down-weighted by a power-law decay. The resulting profile is
    rescaled so that its maximum absolute magnitude does not exceed
    ``max_amplitude``.
    """
    if order < 0:
        raise ValueError("cheb_order must be nonnegative.")
    if max_amplitude < 0.0:
        raise ValueError("Chebyshev amplitude must be nonnegative.")
    if coefficient_decay < 0.0:
        raise ValueError("cheb_coeff_decay must be nonnegative.")
    if max_amplitude == 0.0:
        return np.zeros_like(xi, dtype=float)

    mode_index = np.arange(order + 1, dtype=float)
    coefficient_scale = 1.0 / np.power(mode_index + 1.0, coefficient_decay)
    coefficients = rng.normal(loc=0.0, scale=coefficient_scale)

    perturbation = np.polynomial.chebyshev.chebval(xi, coefficients)
    max_abs = float(np.max(np.abs(perturbation)))
    if max_abs <= 1e-12:
        return np.zeros_like(xi, dtype=float)

    realized_amplitude = rng.uniform(0.35, 1.0) * max_amplitude
    return perturbation * (realized_amplitude / max_abs)


def random_initial_condition_func(
    X: np.ndarray,
    Hbar: np.ndarray,
    Qbar: np.ndarray,
    rng,
    cheb_order: int = CHEB_ORDER_DEFAULT,
    h_cheb_amplitude: float = H_CHEB_AMPLITUDE_DEFAULT,
    q_cheb_amplitude: float = Q_CHEB_AMPLITUDE_DEFAULT,
    cheb_coeff_decay: float = CHEB_COEFF_DECAY_DEFAULT,
) -> Tuple[np.ndarray, np.ndarray]:
    """Generate Chebyshev H and Q initial profiles around the reference state."""
    L = float(X[-1])
    xi = 2.0 * X / L - 1.0

    # Rejection sampling avoids clipping the Chebyshev profiles, because
    # pointwise clipping would destroy their polynomial structure.
    for _ in range(100):
        H_perturbation = sample_truncated_chebyshev(
            xi=xi,
            rng=rng,
            order=cheb_order,
            max_amplitude=h_cheb_amplitude,
            coefficient_decay=cheb_coeff_decay,
        )
        Q_perturbation = sample_truncated_chebyshev(
            xi=xi,
            rng=rng,
            order=cheb_order,
            max_amplitude=q_cheb_amplitude,
            coefficient_decay=cheb_coeff_decay,
        )

        H0 = np.asarray(Hbar, dtype=float) + H_perturbation
        Q0 = np.asarray(Qbar, dtype=float) + Q_perturbation

        finite = np.all(np.isfinite(H0)) and np.all(np.isfinite(Q0))
        positive_depth = float(np.min(H0)) > 0.20
        positive_discharge = float(np.min(Q0)) > 0.0

        if finite and positive_depth and positive_discharge:
            return H0, Q0

    raise RuntimeError(
        "Failed to generate a physically valid Chebyshev initial condition "
        "after 100 attempts. Reduce the Chebyshev amplitudes."
    )


# ============================================================
# 3. Backstepping teacher controller
# ============================================================

def left_eigenvector_matrix(H: float, Q: float, p: CACParams) -> Tuple[np.ndarray, float, float]:
    H = max(float(H), p.H_min)
    V = Q / (p.B * H)
    c = np.sqrt(p.g * H)
    lp = V + c
    lm = V - c

    R = np.array([[1.0, 1.0], [p.B * lp, p.B * lm]], dtype=float)
    T = np.linalg.inv(R)
    return T, lp, lm


def characteristic_variables(
    H: np.ndarray,
    Q: np.ndarray,
    Hbar: np.ndarray,
    p: CACParams,
) -> Tuple[np.ndarray, np.ndarray]:
    u = np.zeros_like(H)
    v = np.zeros_like(H)

    for i in range(len(H)):
        T, _, _ = left_eigenvector_matrix(Hbar[i], p.Q_bar, p)
        z = T @ np.array([H[i] - Hbar[i], Q[i] - p.Q_bar])
        u[i] = z[0]
        v[i] = z[1]

    return u, v


def linearized_matrices_profiles(X: np.ndarray, Hbar: np.ndarray, p: CACParams) -> Dict[str, np.ndarray]:
    N = len(X)
    Qbar = p.Q_bar
    Hx = np.gradient(Hbar, X, edge_order=2)

    T_arr = np.zeros((N, 2, 2))
    R_arr = np.zeros((N, 2, 2))
    A_arr = np.zeros((N, 2, 2))
    lam_plus = np.zeros(N)
    lam_minus = np.zeros(N)

    for i in range(N):
        H = float(Hbar[i])
        Q = Qbar
        T, lp, lm = left_eigenvector_matrix(H, Q, p)
        T_arr[i] = T
        R_arr[i] = np.linalg.inv(T)
        A_arr[i] = np.array(
            [
                [0.0, 1.0 / p.B],
                [p.g * p.B * H - Q**2 / (p.B * H**2), 2.0 * Q / (p.B * H)],
            ],
            dtype=float,
        )
        lam_plus[i] = lp
        lam_minus[i] = lm

    R_x = np.gradient(R_arr, X, axis=0, edge_order=2)

    def G_of_state(Hval: float, Qval: float, Hx_val: float) -> np.ndarray:
        A = np.array(
            [
                [0.0, 1.0 / p.B],
                [p.g * p.B * Hval - Qval**2 / (p.B * Hval**2), 2.0 * Qval / (p.B * Hval)],
            ],
            dtype=float,
        )
        Xbar_x = np.array([Hx_val, 0.0])
        F = source(np.array([Hval]), np.array([Qval]), p)[0]
        return A @ Xbar_x - F

    Bstar = np.zeros((N, 2, 2))
    for i in range(N):
        H = float(Hbar[i])
        Q = Qbar
        hx = float(Hx[i])
        dh = max(1e-6, abs(H) * 1e-6)
        dq = max(1e-6, abs(Q) * 1e-6)
        dG_dH = (G_of_state(H + dh, Q, hx) - G_of_state(H - dh, Q, hx)) / (2.0 * dh)
        dG_dQ = (G_of_state(H, Q + dq, hx) - G_of_state(H, Q - dq, hx)) / (2.0 * dq)
        Bstar[i, :, 0] = dG_dH
        Bstar[i, :, 1] = dG_dQ

    M = np.zeros((N, 2, 2))
    for i in range(N):
        M[i] = T_arr[i] @ (A_arr[i] @ R_x[i] + Bstar[i] @ R_arr[i])

    eps1 = lam_plus.copy()
    eps2 = -lam_minus.copy()

    int1 = np.zeros(N)
    int2 = np.zeros(N)
    f1 = M[:, 0, 0] / eps1
    f2 = -M[:, 1, 1] / eps2
    for i in range(1, N):
        h = X[i] - X[i - 1]
        int1[i] = int1[i - 1] + 0.5 * h * (f1[i] + f1[i - 1])
        int2[i] = int2[i - 1] + 0.5 * h * (f2[i] + f2[i - 1])

    phi1 = np.exp(int1)
    phi2 = np.exp(int2)
    c1 = -(phi1 / phi2) * M[:, 0, 1]
    c2 = -(phi2 / phi1) * M[:, 1, 0]

    Hbar0 = float(Hbar[0])
    kappa0 = -p.U0_bar * p.B * p.mu0 * p.g / np.sqrt(2.0 * p.g * (p.z_up - Hbar0))
    T0 = T_arr[0]
    q0 = (T0[0, 0] + T0[0, 1] * kappa0) / (T0[1, 0] + T0[1, 1] * kappa0)

    return {
        "eps1": eps1,
        "eps2": eps2,
        "eps1_x": np.gradient(eps1, X, edge_order=2),
        "eps2_x": np.gradient(eps2, X, edge_order=2),
        "c1": c1,
        "c2": c2,
        "q0": float(q0),
        "kappa0": float(kappa0),
        "T": T_arr,
        "phi1": phi1,
        "phi2": phi2,
        "M": M,
        "Href": 0.5 * (float(Hbar[0]) + float(Hbar[-1])),
        "Qref": p.Q_bar,
    }


def compute_variable_kernels(X: np.ndarray, coeff: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    N = len(X)
    dx = X[1] - X[0]
    e1 = coeff["eps1"]
    e2 = coeff["eps2"]
    e1x = coeff["eps1_x"]
    e2x = coeff["eps2_x"]
    c1 = coeff["c1"]
    c2 = coeff["c2"]
    q0 = coeff["q0"]
    ratio = q0 * e1[0] / e2[0]

    Kvu = np.zeros((N, N))
    Kvv = np.zeros((N, N))

    for i in range(N):
        Kvu[i, i] = -c2[i] / (e1[i] + e2[i])

    for i in range(1, N):
        A = (e2[i] + e1[0]) / dx - e1x[0]
        rhs = e2[i] * Kvu[i - 1, 0] / dx + e1[0] * Kvu[i, 1] / dx
        denom = A - c2[0] * ratio
        Kvu[i, 0] = rhs / denom
        Kvv[i, 0] = ratio * Kvu[i, 0]

        for j in range(1, i):
            A = (e2[i] + e1[j]) / dx - e1x[j]
            B = (e2[i] + e2[j]) / dx + e2x[j]
            rhsK = e2[i] * Kvu[i - 1, j] / dx + e1[j] * Kvu[i, j + 1] / dx
            rhsL = e2[i] * Kvv[i - 1, j] / dx + e2[j] * Kvv[i, j - 1] / dx
            mat = np.array([[A, -c2[j]], [c1[j], B]], dtype=float)
            sol = np.linalg.solve(mat, np.array([rhsK, rhsL], dtype=float))
            Kvu[i, j] = sol[0]
            Kvv[i, j] = sol[1]

        Bdiag = e2[i] / dx + e2x[i]
        rhsdiag = e2[i] * Kvv[i - 1, i - 1] / dx - c1[i] * Kvu[i, i]
        Kvv[i, i] = rhsdiag / Bdiag

    return Kvu, Kvv


def backstepping_action(
    env: CACSaintVenantPDE1D,
    Kvu_L: np.ndarray,
    Kvv_L: np.ndarray,
    gain: float = 1.0,
) -> Tuple[float, float]:
    p = env.params
    H, Q = env.get_current_HQ()
    Hbar = env.Hbar
    X = env.X_grid

    u, v = characteristic_variables(H, Q, Hbar, p)
    Ubs = gain * float(np.trapz(Kvu_L * u + Kvv_L * v, X))

    T_L, _, _ = left_eigenvector_matrix(float(Hbar[-1]), p.Q_bar, p)
    t21, t22 = T_L[1, 0], T_L[1, 1]

    QL = float(Q[-1])
    Hbs_L = float(Hbar[-1]) + (Ubs - t22 * (QL - p.Q_bar)) / t21
    UL = Hbs_L - p.h_s - (QL**2 / (2.0 * p.g * p.B**2 * p.muL**2)) ** (1.0 / 3.0)
    return float(UL), float(Ubs)


# ============================================================
# 4. State/action helpers
# ============================================================

def make_state_sample(H: np.ndarray, Q: np.ndarray) -> np.ndarray:
    """Return the physical DeepONet input [H, Q] without error normalization."""
    state = np.concatenate([H, Q])
    if not np.all(np.isfinite(state)):
        raise FloatingPointError("Non-finite physical state encountered during collection.")
    return state.astype(np.float32)


def clip_physical_ul(UL: float) -> float:
    return float(np.clip(float(UL), MIN_CONTROL_VALUE, MAX_CONTROL_VALUE))


def physical_ul_to_sac_action(UL: float) -> float:
    UL = clip_physical_ul(UL)
    a = 2.0 * (UL - MIN_CONTROL_VALUE) / (MAX_CONTROL_VALUE - MIN_CONTROL_VALUE) - 1.0
    return float(np.clip(a, -1.0, 1.0))


# ============================================================
# 5. Trajectory-validity checks
# ============================================================

def state_divergence_reason(
    H: np.ndarray,
    Q: np.ndarray,
    p: CACParams,
    max_state_value: float = MAX_STATE_VALUE,
) -> str | None:
    """Return a diagnostic string when the physical state is invalid."""
    H = np.asarray(H, dtype=float)
    Q = np.asarray(Q, dtype=float)

    if H.ndim != 1 or Q.ndim != 1 or H.shape != Q.shape:
        return "invalid_state_shape"
    if not np.all(np.isfinite(H)) or not np.all(np.isfinite(Q)):
        return "non_finite_state"
    if float(np.min(H)) <= float(p.H_min):
        return "water_depth_below_minimum"
    if float(np.max(np.abs(H))) >= float(max_state_value):
        return "water_depth_exceeds_limit"
    if float(np.max(np.abs(Q))) >= float(max_state_value):
        return "discharge_exceeds_limit"
    return None


def info_divergence_reason(info) -> str | None:
    """Read common failure flags from an environment info dictionary."""
    if not isinstance(info, dict):
        return None

    boolean_failure_keys = (
        "diverged",
        "unstable",
        "invalid_state",
        "nan_detected",
        "cfl_violated",
        "pde_failed",
        "integration_failed",
    )
    for key in boolean_failure_keys:
        if bool(info.get(key, False)):
            return f"info_{key}"

    for key in ("reason", "termination_reason", "truncation_reason"):
        value = info.get(key)
        if value is None:
            continue
        value_text = str(value).strip().lower()
        if any(token in value_text for token in (
            "diverg", "unstable", "invalid", "nan", "inf", "cfl", "fail", "overflow"
        )):
            return f"{key}:{value_text}"

    return None


def normal_horizon_reached(env, T: float, control_sample_rate: float) -> bool:
    """Allow a small tolerance because the final control step can overshoot T."""
    final_time = float(getattr(env, "time", 0.0))
    tolerance = max(1e-9, 0.5 * float(control_sample_rate))
    return final_time >= float(T) - tolerance


# ============================================================
# 6. Dataset generation
# ============================================================

def datasetGenerate(
    num: int = 1000,
    T: float = T_DEFAULT,
    dt: float = DT_DEFAULT,
    N: int = N_DEFAULT,
    control_sample_rate: float = CONTROL_SAMPLE_RATE_DEFAULT,
    out_dir: str | Path = "./dataset_chebyshev_ODE_AH03_AQ25",
    reference: str = "ode",
    q_bar: float = Q_BAR_FIXED_DEFAULT,
    random_seed: int | None = None,
    cheb_order: int = CHEB_ORDER_DEFAULT,
    h_cheb_amplitude: float = H_CHEB_AMPLITUDE_DEFAULT,
    q_cheb_amplitude: float = Q_CHEB_AMPLITUDE_DEFAULT,
    cheb_coeff_decay: float = CHEB_COEFF_DECAY_DEFAULT,
    max_attempt_factor: int = MAX_ATTEMPT_FACTOR_DEFAULT,
) -> None:
    """Collect exactly ``num`` valid trajectories whenever possible.

    Each episode is first cached in memory. Files are updated only after the
    episode reaches the requested horizon and passes all validity checks. If
    the trajectory diverges at any point, all samples from that episode are
    discarded.
    """
    if num <= 0:
        raise ValueError("num must be positive.")
    if max_attempt_factor < 1:
        raise ValueError("max_attempt_factor must be at least 1.")

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(random_seed)

    if not np.isfinite(q_bar) or q_bar <= 0.0:
        raise ValueError("q_bar must be a positive finite number.")

    nominal_p = make_cac_table_params(q_bar=q_bar)
    X, dx = make_grid(nominal_p, N)
    Hbar, Qbar, Q_equilibrium = make_reference_profiles(
        X, nominal_p, reference=reference, q_bar_fixed=q_bar
    )

    # Recreate the parameter object so every environment and controller
    # component uses the same prescribed reference discharge and ODE depth.
    p = make_cac_table_params(
        q_bar=Q_equilibrium,
        h_bar_0=float(Hbar[0]),
        h_bar_L=float(Hbar[-1]),
    )
    H_init, Q_init = Hbar.copy(), Qbar.copy()

    coeff = linearized_matrices_profiles(X, Hbar, p)
    Kvu, Kvv = compute_variable_kernels(X, coeff)
    Kvu_L = Kvu[-1, :]
    Kvv_L = Kvv[-1, :]

    def chebyshev_reset_func(X_reset, Hbar_reset, Qbar_reset, rng_reset):
        return random_initial_condition_func(
            X=X_reset,
            Hbar=Hbar_reset,
            Qbar=Qbar_reset,
            rng=rng_reset,
            cheb_order=cheb_order,
            h_cheb_amplitude=h_cheb_amplitude,
            q_cheb_amplitude=q_cheb_amplitude,
            cheb_coeff_decay=cheb_coeff_decay,
        )

    env_kwargs = dict(
        params=p,
        X_grid=X,
        Hbar=Hbar,
        Qbar=Qbar,
        H_init=H_init,
        Q_init=Q_init,
        sensing_noise_func=lambda state: state,
        reset_init_condition_func=chebyshev_reset_func,
        sensing_loc="full",
        control_type="Gate",
        limit_pde_state_size=True,
        max_state_value=MAX_STATE_VALUE,
        min_control_value=MIN_CONTROL_VALUE,
        max_control_value=MAX_CONTROL_VALUE,
        control_sample_rate=control_sample_rate,
        reward_class=None,
        normalize=False,
        observation_mode="normalized_error",
        H_scale=H_SCALE,
        Q_scale=Q_SCALE,
        obs_clip=OBS_CLIP,
        T=T,
        dt=dt,
        cfl=CFL,
        check_cfl=CHECK_CFL,
        well_balanced=WELL_BALANCED,
    )

    x_path = out_dir / "x.dat"
    y_path = out_dir / "y.dat"
    y_ul_path = out_dir / "y_ul.dat"
    y_ul_raw_path = out_dir / "y_ul_raw.dat"
    y_ubs_path = out_dir / "y_ubs.dat"
    meta_path = out_dir / "meta.dat"
    rejected_path = out_dir / "rejected_episodes.dat"
    grid_path = out_dir / "grid.dat"
    ref_path = out_dir / "hbar_qbar.dat"

    np.savetxt(grid_path, X)
    np.savetxt(ref_path, np.concatenate([Hbar, Qbar])[None, :])

    for path in [
        x_path,
        y_path,
        y_ul_path,
        y_ul_raw_path,
        y_ubs_path,
        meta_path,
        rejected_path,
    ]:
        if path.exists():
            path.unlink()

    max_attempts = int(num) * int(max_attempt_factor)
    pbar = tqdm(total=int(num), desc="Accepted Chebyshev trajectories") if tqdm is not None else None

    total_samples = 0
    accepted_episodes = 0
    rejected_episodes = 0
    attempt_id = 0
    with open(x_path, "ab") as fx, \
         open(y_path, "ab") as fy, \
         open(y_ul_path, "ab") as fyul, \
         open(y_ul_raw_path, "ab") as fyulraw, \
         open(y_ubs_path, "ab") as fubs, \
         open(meta_path, "ab") as fm, \
         open(rejected_path, "w", encoding="utf-8") as frej:

        frej.write("attempt_id\tseed\tfailure_time_s\tH_min\tH_max\tQ_min\tQ_max\treason\n")

        while accepted_episodes < num and attempt_id < max_attempts:
            current_attempt = attempt_id
            attempt_id += 1
            seed = int(rng.integers(0, 2**31 - 1))

            x_episode = []
            y_episode = []
            y_ul_episode = []
            y_ul_raw_episode = []
            y_ubs_episode = []
            meta_episode = []

            diverged = False
            failure_reason = ""
            failure_time = 0.0
            diagnostic = [np.nan, np.nan, np.nan, np.nan]

            env = None
            try:
                env = CACSaintVenantPDE1D(**env_kwargs)
                obs, reset_info = env.reset(seed=seed)

                H, Q = env.get_current_HQ()
                failure_reason = state_divergence_reason(H, Q, p) or ""
                failure_reason = failure_reason or (info_divergence_reason(reset_info) or "")
                if failure_reason:
                    diverged = True

                terminate = False
                truncate = False

                while not diverged and not (terminate or truncate):
                    H, Q = env.get_current_HQ()
                    failure_reason = state_divergence_reason(H, Q, p) or ""
                    if failure_reason:
                        diverged = True
                        break

                    x_sample = make_state_sample(H, Q)

                    UL_raw, Ubs = backstepping_action(env, Kvu_L, Kvv_L, gain=1.0)
                    if not np.isfinite(UL_raw) or not np.isfinite(Ubs):
                        failure_reason = "non_finite_control"
                        diverged = True
                        break

                    UL_clip = clip_physical_ul(UL_raw)
                    a_sac = physical_ul_to_sac_action(UL_clip)

                    # Cache only. Nothing is written before the full episode is accepted.
                    x_episode.append(x_sample)
                    y_episode.append([a_sac])
                    y_ul_episode.append([UL_clip])
                    y_ul_raw_episode.append([UL_raw])
                    y_ubs_episode.append([float(Ubs)])
                    meta_episode.append([accepted_episodes, float(env.time)])

                    obs, reward, terminate, truncate, info = env.step(
                        np.array([UL_clip], dtype=np.float32)
                    )

                    if not np.isfinite(reward):
                        failure_reason = "non_finite_reward"
                        diverged = True
                        break

                    info_reason = info_divergence_reason(info)
                    if info_reason:
                        failure_reason = info_reason
                        diverged = True
                        break

                    H_next, Q_next = env.get_current_HQ()
                    failure_reason = state_divergence_reason(H_next, Q_next, p) or ""
                    if failure_reason:
                        diverged = True
                        break

                    # An early terminal/truncation is treated as a failed trajectory.
                    if (terminate or truncate) and not normal_horizon_reached(
                        env, T=T, control_sample_rate=control_sample_rate
                    ):
                        failure_reason = "episode_ended_before_target_horizon"
                        diverged = True
                        break

                if not diverged and not normal_horizon_reached(
                    env, T=T, control_sample_rate=control_sample_rate
                ):
                    failure_reason = "target_horizon_not_reached"
                    diverged = True

            except Exception as exc:
                diverged = True
                failure_reason = f"exception:{type(exc).__name__}:{str(exc)[:160]}"

            if env is not None:
                failure_time = float(getattr(env, "time", 0.0))
                try:
                    H_diag, Q_diag = env.get_current_HQ()
                    diagnostic = [
                        float(np.nanmin(H_diag)),
                        float(np.nanmax(H_diag)),
                        float(np.nanmin(Q_diag)),
                        float(np.nanmax(Q_diag)),
                    ]
                except Exception:
                    pass

            if diverged:
                rejected_episodes += 1
                safe_reason = failure_reason.replace("\t", " ").replace("\n", " ")
                frej.write(
                    f"{current_attempt}\t{seed}\t{failure_time:.9g}\t"
                    f"{diagnostic[0]:.9g}\t{diagnostic[1]:.9g}\t"
                    f"{diagnostic[2]:.9g}\t{diagnostic[3]:.9g}\t{safe_reason}\n"
                )
                frej.flush()
                print(
                    f"Discard trajectory attempt={current_attempt}, seed={seed}, "
                    f"t={failure_time:.3f}s, reason={failure_reason}"
                )
                continue

            if len(x_episode) == 0:
                rejected_episodes += 1
                frej.write(
                    f"{current_attempt}\t{seed}\t{failure_time:.9g}\t"
                    f"{diagnostic[0]:.9g}\t{diagnostic[1]:.9g}\t"
                    f"{diagnostic[2]:.9g}\t{diagnostic[3]:.9g}\tempty_trajectory\n"
                )
                frej.flush()
                print(f"Discard empty trajectory attempt={current_attempt}, seed={seed}")
                continue

            # Commit the complete valid trajectory atomically at episode level.
            np.savetxt(fx, np.asarray(x_episode, dtype=np.float32))
            np.savetxt(fy, np.asarray(y_episode, dtype=np.float32))
            np.savetxt(fyul, np.asarray(y_ul_episode, dtype=np.float32))
            np.savetxt(fyulraw, np.asarray(y_ul_raw_episode, dtype=np.float32))
            np.savetxt(fubs, np.asarray(y_ubs_episode, dtype=np.float32))
            np.savetxt(fm, np.asarray(meta_episode, dtype=np.float32))

            total_samples += len(x_episode)
            accepted_episodes += 1
            if pbar is not None:
                pbar.update(1)

    if pbar is not None:
        pbar.close()

    if accepted_episodes < num:
        raise RuntimeError(
            f"Only {accepted_episodes}/{num} valid trajectories were collected after "
            f"{attempt_id} attempts. Increase --max-attempt-factor or reduce amplitudes."
        )

    print("Dataset generation finished.")
    print("Date:", datetime.datetime.now())
    print(f"Saved x.dat: {x_path}")
    print(f"Saved y.dat: {y_path}")
    print(f"Saved y_ul.dat: {y_ul_path}")
    print(f"Saved y_ul_raw.dat: {y_ul_raw_path}")
    print(f"Saved y_ubs.dat: {y_ubs_path}")
    print(f"Saved meta.dat: {meta_path}")
    print(f"Saved rejected_episodes.dat: {rejected_path}")
    print(f"Saved grid.dat: {grid_path}")
    print(f"Saved hbar_qbar.dat: {ref_path}")
    print(f"requested valid episodes = {num}")
    print(f"accepted episodes = {accepted_episodes}")
    print(f"rejected divergent episodes = {rejected_episodes}")
    print(f"total attempts = {attempt_id}")
    print(f"total accepted samples = {total_samples}")
    print(f"x dimension per sample = {2 * N}")
    print("y dimension per sample = 1")
    print(f"N = {N}, dx = {dx:.6f} m, T = {T} s, dt = {dt} s")
    print(f"control_sample_rate = {control_sample_rate} s")
    print(f"reference profile method = {reference}")
    print(f"ODE/reference Hbar(0) = {Hbar[0]:.9f} m")
    print(f"ODE/reference Hbar(L) = {Hbar[-1]:.9f} m")
    print(f"fixed Qbar = {Qbar[0]:.9f} m^3/s")
    print(f"max spatial variation of Qbar = {np.ptp(Qbar):.3e} m^3/s")
    print(
        "Chebyshev IC settings: "
        f"order={cheb_order}, "
        f"A_H={h_cheb_amplitude} m, "
        f"A_Q={q_cheb_amplitude} m^3/s, "
        f"coefficient decay={cheb_coeff_decay}"
    )
    print("Divergent trajectories are discarded in full before any sample is written.")
    print(f"physical U_L range = [{MIN_CONTROL_VALUE}, {MAX_CONTROL_VALUE}]")
    print("x.dat = physical state concat([H, Q]); no error normalization is applied")
    print("rollout uses physical U_L because env normalize=False")
    print("y.dat = normalized SAC action label a in [-1,1]")
    print("y_ul.dat = physical gate label in metres")

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--num", type=int, default=1000, help="number of episodes")
    parser.add_argument("--N", type=int, default=N_DEFAULT)
    parser.add_argument("--T", type=float, default=T_DEFAULT)
    parser.add_argument("--dt", type=float, default=DT_DEFAULT)
    parser.add_argument("--control-sample-rate", type=float, default=CONTROL_SAMPLE_RATE_DEFAULT)
    parser.add_argument("--out-dir", type=str, default="./dataset_chebyshev_fixed_Qbar_11p93_ODE_Hbar_AH03_AQ25")
    parser.add_argument(
        "--reference",
        choices=["ode", "strict", "linear"],
        default="ode",
        help="Hbar reference: steady ODE by default; Qbar remains fixed",
    )
    parser.add_argument(
        "--q-bar",
        type=float,
        default=Q_BAR_FIXED_DEFAULT,
        help="fixed spatially constant reference discharge in m^3/s",
    )
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument(
        "--cheb-order",
        type=int,
        default=CHEB_ORDER_DEFAULT,
        help="highest retained Chebyshev order",
    )
    parser.add_argument(
        "--h-cheb-amplitude",
        type=float,
        default=H_CHEB_AMPLITUDE_DEFAULT,
        help="maximum interior H perturbation in metres",
    )
    parser.add_argument(
        "--q-cheb-amplitude",
        type=float,
        default=Q_CHEB_AMPLITUDE_DEFAULT,
        help="maximum interior Q perturbation in m^3/s",
    )
    parser.add_argument(
        "--cheb-coeff-decay",
        type=float,
        default=CHEB_COEFF_DECAY_DEFAULT,
        help="power-law decay exponent for higher Chebyshev modes",
    )
    parser.add_argument(
        "--max-attempt-factor",
        type=int,
        default=MAX_ATTEMPT_FACTOR_DEFAULT,
        help="maximum attempted trajectories divided by requested valid trajectories",
    )
    args, _unknown = parser.parse_known_args()

    datasetGenerate(
        num=args.num,
        T=args.T,
        dt=args.dt,
        N=args.N,
        control_sample_rate=args.control_sample_rate,
        out_dir=args.out_dir,
        reference=args.reference,
        q_bar=args.q_bar,
        random_seed=args.seed,
        cheb_order=args.cheb_order,
        h_cheb_amplitude=args.h_cheb_amplitude,
        q_cheb_amplitude=args.q_cheb_amplitude,
        cheb_coeff_decay=args.cheb_coeff_decay,
        max_attempt_factor=args.max_attempt_factor,
    )


if __name__ == "__main__":
    main()


In [ ]:
# -*- coding: utf-8 -*-

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from tqdm import trange
from timeit import default_timer


# ============================================================
# 1. 基本参数
# ============================================================
epochs = 400
ntrain = 900
ntest = 100

gamma = 0.5
learning_rate = 0.0001
step_size = 40

modes = 12
width = 32
batch_size = 50
sub_xy = 1

# Branch和Trunk的输出特征维数
latent_dim = 102

# CAC数据文件
# x.dat 每一行：
# [H_1, ..., H_N, Q_1, ..., Q_N]
x_file = "./dataset_chebyshev_fixed_Qbar_11p93_ODE_Hbar_AH03_AQ25/x.dat"

# y.dat 每一行：
# 下游边界控制量 U_L(t)
y_file = "./dataset_chebyshev_fixed_Qbar_11p93_ODE_Hbar_AH03_AQ25/y_ul.dat"

# 由于网络结构已经改变，建议使用新的模型文件名，
# 避免覆盖原来采用点积结构的模型。
model_name = "modedlNet10000QqQqQ14444_hadamard.pt"

# 训练损失保存路径
train_loss_file = "train_loss_hadamard.dat"
test_loss_file = "test_loss_hadamard.dat"
time_file = "time_Arr_hadamard.dat"

# 损失曲线图片
loss_figure_name = "trainLoss10000QqQqQ14444_hadamard.jpg"


# ============================================================
# 2. 计算设备
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print(f"当前计算设备：{device}")
print("=" * 70)


# ============================================================
# 3. 加载输入数据x.dat和标签数据y.dat
# ============================================================
inpArr1 = np.loadtxt(x_file, dtype=np.float32)
outArr = np.loadtxt(y_file, dtype=np.float32)

# 只有一个样本时，将其恢复为二维数组
if inpArr1.ndim == 1:
    inpArr1 = inpArr1.reshape(1, -1)

# 按照sub_xy进行采样
# xxx形状：[样本数, 2N]
xxx = inpArr1[::sub_xy, :]

# yyy形状：[样本数]
yyy = outArr[::sub_xy].reshape(-1)

print(f"xxx形状：{xxx.shape}")
print(f"yyy形状：{yyy.shape}")

# 检查输入和输出样本数量是否一致
if xxx.shape[0] != yyy.shape[0]:
    raise ValueError(
        f"x.dat和y.dat的样本数不一致："
        f"x样本数={xxx.shape[0]}，y样本数={yyy.shape[0]}"
    )

# CAC状态由H和Q组成，因此列数应当为偶数
if xxx.shape[1] % 2 != 0:
    raise ValueError(
        "x.dat的列数必须是偶数，"
        "因为每一行应当为[H_1,...,H_N,Q_1,...,Q_N]。"
    )

# 推断空间网格点数量N
nx = xxx.shape[1] // 2

print(f"推断网格点数 nx = {nx}")
print(f"系统状态输入维数 = {2 * nx}")


# ============================================================
# 4. 划分训练集和测试集
# ============================================================
x_train_np, x_test_np, y_train_np, y_test_np = train_test_split(
    xxx,
    yyy,
    test_size=0.1,
    random_state=1
)

# 转换为PyTorch张量
x_train = torch.from_numpy(x_train_np).float().to(device)
y_train = torch.from_numpy(y_train_np).float().to(device)

x_test = torch.from_numpy(x_test_np).float().to(device)
y_test = torch.from_numpy(y_test_np).float().to(device)

print("\nx_train前5行数据：")
print(x_train[:5].detach().cpu().numpy())

print("\ny_train前5个数据：")
print(y_train[:5].detach().cpu().numpy())

print("\nx_test前5行数据：")
print(x_test[:5].detach().cpu().numpy())

print("\ny_test前5个数据：")
print(y_test[:5].detach().cpu().numpy())

print(f"\nx_train形状：{x_train.shape}")
print(f"y_train形状：{y_train.shape}")
print(f"x_test形状：{x_test.shape}")
print(f"y_test形状：{y_test.shape}")


# ============================================================
# 5. 构造DataLoader
# ============================================================
trainData = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=batch_size,
    shuffle=True
)

testData = DataLoader(
    TensorDataset(x_test, y_test),
    batch_size=batch_size,
    shuffle=False
)


# ============================================================
# 6. DeepONet的Trunk输入
#
# 当前DeepONet学习：
#     [H(t,x), Q(t,x)] -> U_L(t)
#
# U_L是下游边界控制量，因此查询位置固定为归一化坐标x=1。
#
# grid2形状：
#     [1, 1]
# ============================================================
grid_1d = np.array([[1.0]], dtype=np.float32)
grid2 = torch.from_numpy(grid_1d).float().to(device)

print(f"\ngrid2形状：{grid2.shape}")
print(f"固定查询位置：x = {grid2.item()}")


# ============================================================
# 7. Branch网络
# ============================================================
class BranchNet(nn.Module):
    """
    Branch网络用于提取Saint-Venant系统的分布式状态特征。

    输入：
        x.shape = [batch, 2N]

    状态排列：
        [H_1, ..., H_N, Q_1, ..., Q_N]

    输出：
        b(s).shape = [batch, latent_dim]
    """

    def __init__(self, shape, output_dim=102):
        super().__init__()

        self.shape = int(shape)
        self.output_dim = int(output_dim)

        # 输入通道：
        # 通道1为H，通道2为Q
        self.conv1 = nn.Conv1d(
            in_channels=2,
            out_channels=16,
            kernel_size=5,
            stride=2
        )

        self.relu = nn.ReLU()

        self.conv2 = nn.Conv1d(
            in_channels=16,
            out_channels=32,
            kernel_size=5,
            stride=2
        )

        # 自动计算卷积层输出展开后的维数
        with torch.no_grad():
            temp = torch.zeros(1, 2, self.shape)

            temp = self.conv1(temp)
            temp = self.relu(temp)

            temp = self.conv2(temp)
            temp = self.relu(temp)

            flat_dim = temp.reshape(1, -1).shape[1]

        # 将卷积特征映射到latent_dim维
        self.fc1 = nn.Linear(flat_dim, self.output_dim)

    def forward(self, x):
        """
        参数
        ----
        x：
            [batch, 2N]

        返回
        ----
        b：
            Branch状态特征，[batch, latent_dim]
        """

        if x.dim() != 2:
            raise ValueError(
                f"Branch输入必须是二维张量[batch, 2N]，"
                f"当前形状为{x.shape}"
            )

        if x.shape[1] != 2 * self.shape:
            raise ValueError(
                f"Branch输入维数错误，期望{2 * self.shape}，"
                f"实际为{x.shape[1]}"
            )

        # [batch, 2N] -> [batch, 2, N]
        x = x.reshape(x.shape[0], 2, self.shape)

        # 第一层卷积
        x = self.conv1(x)
        x = self.relu(x)

        # 第二层卷积
        x = self.conv2(x)
        x = self.relu(x)

        # 展平
        x = torch.flatten(x, start_dim=1)

        # 全连接状态编码
        x = self.fc1(x)
        x = self.relu(x)

        # x就是Branch输出b(s)
        return x


# ============================================================
# 8. Trunk网络
# ============================================================
class TrunkNet(nn.Module):
    """
    Trunk网络用于编码空间查询位置。

    输入：
        grid.shape = [查询点数量, dim_x]

    当前只有一个查询位置x=1，因此：
        grid.shape = [1, 1]

    输出：
        t(x).shape = [查询点数量, latent_dim]
    """

    def __init__(self, dim_x=1, output_dim=102):
        super().__init__()

        self.dim_x = int(dim_x)
        self.output_dim = int(output_dim)

        self.net = nn.Sequential(
            nn.Linear(self.dim_x, 16),
            nn.ReLU(),

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, self.output_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# 9. 采用逐元素融合的DeepONet
# ============================================================
class DeepONetModified(nn.Module):
    """
    采用Hadamard逐元素乘积进行Branch-Trunk特征融合。

    Branch输出：
        b(s) ∈ R^latent_dim

    Trunk输出：
        t(x) ∈ R^latent_dim

    融合特征：
        z(s,x) = b(s) ⊙ t(x) ∈ R^latent_dim

    控制输出：
        U_L = f_ctrl(z) ∈ R^projection

    注意：
        这里不再使用普通点积b(s)^T t(x)，
        因此融合结果z仍然是一个latent_dim维特征向量。
    """

    def __init__(
        self,
        m2,
        branch_2dim,
        dim_x,
        projection,
        latent_dim=102
    ):
        super().__init__()

        self.m2 = int(m2)
        self.latent_dim = int(latent_dim)
        self.projection = int(projection)

        # Branch状态特征提取器
        self.branch = branch_2dim

        # Trunk空间位置编码器
        self.trunk = TrunkNet(
            dim_x=dim_x,
            output_dim=self.latent_dim
        )

        # 融合后控制输出头
        #
        # 原来的点积输出是1维：
        #     Linear(1, 32)
        #
        # 现在逐元素融合输出是latent_dim维：
        #     Linear(latent_dim, 32)
        self.fc3 = nn.Linear(self.latent_dim, 32)
        self.relu = nn.ReLU()
        self.fc4 = nn.Linear(32, 16)
        self.fc5 = nn.Linear(16, self.projection)

    def extract_branch_trunk_features(self, x_state, grid):
        """
        分别计算Branch和Trunk特征。

        返回
        ----
        b：
            Branch状态特征，[batch, latent_dim]

        t：
            与batch维度匹配后的Trunk空间特征，
            [batch, latent_dim]
        """

        # Branch状态特征
        b = self.branch(x_state)

        # Trunk空间特征
        t = self.trunk(grid)

        # 当前grid只有一个固定查询点x=1，
        # 因此t原始形状通常为[1, latent_dim]。
        #
        # 将其扩展为[batch, latent_dim]，
        # 使每个状态样本都与相同的下游位置特征融合。
        if t.shape[0] == 1:
            t = t.expand(b.shape[0], -1)

        # 如果Trunk输入本身与batch一一对应，则直接使用
        elif t.shape[0] == b.shape[0]:
            pass

        else:
            raise ValueError(
                "Branch和Trunk的batch维度不匹配："
                f"b.shape={b.shape}，t.shape={t.shape}"
            )

        if b.shape[1] != t.shape[1]:
            raise ValueError(
                "Branch和Trunk的特征维数不一致："
                f"b.shape={b.shape}，t.shape={t.shape}"
            )

        return b, t

    def extract_latent(self, x_state, grid):
        """
        提取Branch-Trunk逐元素融合后的潜在特征。

        z = b(s) ⊙ t(x)

        返回：
            z.shape = [batch, latent_dim]
        """

        b, t = self.extract_branch_trunk_features(
            x_state,
            grid
        )

        # Hadamard逐元素乘积
        z = b * t

        return z

    def control_head(self, z):
        """
        将latent_dim维融合特征映射为控制标量。

        latent_dim -> 32 -> 16 -> projection
        """

        x = self.fc3(z)
        x = self.relu(x)

        x = self.fc4(x)
        x = self.relu(x)

        control = self.fc5(x)

        return control

    def forward(self, x, return_latent=False):
        """
        默认只返回控制输出。

        当return_latent=True时，同时返回：
            control：
                控制输出，[batch, projection]

            z：
                逐元素融合特征，[batch, latent_dim]
        """

        x_state, grid = x[0], x[1]

        # 计算融合潜在特征
        z = self.extract_latent(
            x_state,
            grid
        )

        # 融合特征经过控制头输出控制量
        control = self.control_head(z)

        if return_latent:
            return control, z

        return control


# ============================================================
# 10. 创建模型
# ============================================================
m2 = nx
projection = 1

branch_network = BranchNet(
    shape=nx,
    output_dim=latent_dim
)

model = DeepONetModified(
    m2=m2,
    branch_2dim=branch_network,
    dim_x=1,
    projection=projection,
    latent_dim=latent_dim
).to(device)

print("\n模型结构：")
print(model)

# 查看可训练参数数量
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"\n模型总参数量：{total_params}")
print(f"可训练参数量：{trainable_params}")


# ============================================================
# 11. 优化器和学习率调度器
# ============================================================
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=step_size,
    gamma=gamma
)

# 使用SmoothL1损失
loss_function = nn.SmoothL1Loss(
    reduction="mean",
    beta=1.0
)

train_lossArr = []
test_lossArr = []
time_Arr = []


# ============================================================
# 12. 训练前检查网络输出维数
# ============================================================
model.eval()

with torch.no_grad():
    check_batch_size = min(5, x_train.shape[0])

    check_control, check_z = model(
        (x_train[:check_batch_size], grid2),
        return_latent=True
    )

print("\n训练前维数检查：")
print(
    f"输入状态形状："
    f"{x_train[:check_batch_size].shape}"
)
print(
    f"逐元素融合特征z形状："
    f"{check_z.shape}"
)
print(
    f"控制输出形状："
    f"{check_control.shape}"
)

expected_z_shape = (
    check_batch_size,
    latent_dim
)

expected_control_shape = (
    check_batch_size,
    projection
)

if tuple(check_z.shape) != expected_z_shape:
    raise RuntimeError(
        f"融合特征z形状错误，期望{expected_z_shape}，"
        f"实际为{tuple(check_z.shape)}"
    )

if tuple(check_control.shape) != expected_control_shape:
    raise RuntimeError(
        f"控制输出形状错误，期望{expected_control_shape}，"
        f"实际为{tuple(check_control.shape)}"
    )


# ============================================================
# 13. 训练
# ============================================================
print("\n开始训练")
print("=" * 70)

for ep in trange(epochs):
    start_time = default_timer()

    # --------------------------------------------------------
    # 训练阶段
    # --------------------------------------------------------
    model.train()
    train_loss = 0.0

    for xxTemp, yyTemp in trainData:
        xx = xxTemp.to(device)
        y1 = yyTemp.to(device)

        optimizer.zero_grad()

        # 输出控制量
        out = model((xx, grid2))

        # out形状：[当前batch, 1]
        # y1形状：[当前batch]
        #
        # 将二者均转换为[当前batch, 1]
        current_loss = loss_function(
            out.reshape(out.shape[0], -1),
            y1.reshape(y1.shape[0], -1)
        )

        current_loss.backward()

        optimizer.step()

        train_loss += current_loss.item()

    # 每个epoch结束后更新学习率
    scheduler.step()

    # --------------------------------------------------------
    # 测试阶段
    # --------------------------------------------------------
    model.eval()
    test_loss = 0.0

    with torch.no_grad():
        for xxTemp, yyTemp in testData:
            xx = xxTemp.to(device)
            y1 = yyTemp.to(device)

            out = model((xx, grid2))

            current_test_loss = loss_function(
                out.reshape(out.shape[0], -1),
                y1.reshape(y1.shape[0], -1)
            )

            test_loss += current_test_loss.item()

    # 计算每个batch的平均损失
    train_loss /= len(trainData)
    test_loss /= len(testData)

    train_lossArr.append(train_loss)
    test_lossArr.append(test_loss)

    end_time = default_timer()
    epoch_time = end_time - start_time
    time_Arr.append(epoch_time)

    # 每50个epoch输出一次结果
    if ep % 50 == 0:
        current_lr = optimizer.param_groups[0]["lr"]

        print("\n" + "-" * 70)
        print(f"Epoch：{ep}")
        print(f"本轮训练时间：{epoch_time:.6f} s")
        print(f"当前学习率：{current_lr:.8e}")
        print(f"训练损失：{train_loss:.8e}")
        print(f"测试损失：{test_loss:.8e}")

        with torch.no_grad():
            sample_num = min(5, x_test.shape[0])

            pred, latent_z = model(
                (x_test[:sample_num], grid2),
                return_latent=True
            )

            pred = (
                pred.detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            true = (
                y_test[:sample_num]
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

        print("预测U_L前5个：", pred)
        print("真实U_L前5个：", true)
        print("融合特征z形状：", latent_z.shape)

print("\n训练结束")
print("=" * 70)


# ============================================================
# 14. 保存模型参数
# ============================================================
torch.save(
    model.state_dict(),
    model_name
)

print(f"模型参数已保存：{model_name}")

np.savetxt(
    train_loss_file,
    np.asarray(train_lossArr)
)

np.savetxt(
    test_loss_file,
    np.asarray(test_lossArr)
)

np.savetxt(
    time_file,
    np.asarray(time_Arr)
)

print(f"训练损失已保存：{train_loss_file}")
print(f"测试损失已保存：{test_loss_file}")
print(f"训练时间已保存：{time_file}")


# ============================================================
# 15. 最终测试结果
# ============================================================
model.eval()

with torch.no_grad():
    sample_num = min(5, x_test.shape[0])

    final_prediction, final_latent = model(
        (x_test[:sample_num], grid2),
        return_latent=True
    )

final_prediction = (
    final_prediction
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

final_true = (
    y_test[:sample_num]
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

final_latent_np = (
    final_latent
    .detach()
    .cpu()
    .numpy()
)

print("\n最终测试结果：")
print("预测控制量：", final_prediction)
print("真实控制量：", final_true)
print("融合特征z形状：", final_latent_np.shape)

# 可选：保存前5个样本的融合特征
np.savetxt(
    "latent_features_hadamard.dat",
    final_latent_np
)

print(
    "前5个测试样本的融合特征已保存："
    "latent_features_hadamard.dat"
)


# ============================================================
# 16. 绘制损失曲线
# ============================================================
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.plot(
    train_lossArr,
    label="Train Loss",
    linewidth=2
)

ax.plot(
    test_lossArr,
    label="Test Loss",
    linewidth=2
)

ax.set_xlabel(
    "Epoch",
    fontsize=13
)

ax.set_ylabel(
    "Smooth L1 Loss",
    fontsize=13
)

ax.tick_params(
    labelsize=10
)

ax.set_yscale("log")

ax.legend(
    fontsize=13,
    loc="best"
)

fig.tight_layout()

plt.savefig(
    loss_figure_name,
    dpi=500,
    bbox_inches="tight"
)

print(f"损失曲线已保存：{loss_figure_name}")

plt.show()